# INIT

In [1]:
from pyprojroot import here
import warnings
from pathlib import Path
from typing import Tuple, Dict, List

import json
import numpy as np
import pandas as pd
import cv2
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
import timm

from monai.data import CacheDataset, DataLoader
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    Resized,
    Flipd,
    Rotate90d,
    RandFlipd,
    RandRotate90d,
    RandRotated,
    RandZoomd,
    RandGaussianNoised,
    RandAdjustContrastd,
    RandGaussianSmoothd,
    RandAffined,
    NormalizeIntensityd,
    ToTensord,
    MapTransform,
)
from monai.utils import set_determinism
from monai.config import KeysCollection

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

warnings.filterwarnings("ignore")

In [2]:
class Config:
    # ==================== Paths ====================
    PARENTS = here()
    DATA_DIR = PARENTS / Path("data")
    LABELS_CSV = DATA_DIR / "Labels.csv"
    IMAGES_DIR = DATA_DIR / "Images"
    OUTPUT_DIR = PARENTS / Path("outputs")
    FINAL_OUTPUT_DIR = PARENTS / Path("outputs/final")

    # ==================== Dataset ====================
    IMG_SIZE = 380
    LABEL_COL = "Label"
    IMAGE_COL = "Image Name"
    PATIENT_COL = "Patient"
    QUALITY_COL = "Quality Score"

    # ==================== Training Params ====================
    SEED = 42
    BATCH_SIZE = 16
    NUM_WORKERS = 4
    EPOCHS = 40
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.4
    EARLY_STOP = 8

    # ==================== Augmentation ====================
    USE_MIXUP = True
    MIXUP_ALPHA = 0.2

    # ==================== Model ====================
    BACKBONE = "efficientnet_b3"
    PRETRAINED = True

    # ==================== Inference ====================
    USE_TTA = True
    TTA_STEPS = 5

    # ==================== Classes ====================
    CLASS_NAMES = ["GON-", "GON+"]
    POS_LABEL = "GON+"

    # ==================== MONAI CacheDataset ====================
    CACHE_RATE = 1.0

    def __init__(self):
        self.DATA_DIR.mkdir(parents=True, exist_ok=True)
        self.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        self.FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_determinism(seed=cfg.SEED)

# Print path to make sure it correct
print(f"Project root : {cfg.PARENTS}")
print(f"Labels CSV   : {cfg.LABELS_CSV}")
print(f"Images dir   : {cfg.IMAGES_DIR}")
print(f"Output dir   : {cfg.OUTPUT_DIR}")
print()
print(f"Labels dir exists : {cfg.LABELS_CSV.exists()}")
print(f"Images dir exists : {cfg.IMAGES_DIR.exists()}")

Project root : c:\Users\LMS\Documents\CodeProject\Py\HYGD Dataset
Labels CSV   : c:\Users\LMS\Documents\CodeProject\Py\HYGD Dataset\data\Labels.csv
Images dir   : c:\Users\LMS\Documents\CodeProject\Py\HYGD Dataset\data\Images
Output dir   : c:\Users\LMS\Documents\CodeProject\Py\HYGD Dataset\outputs

Labels dir exists : True
Images dir exists : True


# Load dataset

In [3]:
def load_and_clean_labels(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    print(f"[DATA] Raw: {len(df)} images, {df[cfg.PATIENT_COL].nunique()} patients")
    print(f"Classes:\n{df[cfg.LABEL_COL].value_counts()}\n")

    # binary encoding
    df["label"] = (df[cfg.LABEL_COL] == cfg.POS_LABEL).astype(int)
    df["image_path"] = df[cfg.IMAGE_COL].apply(lambda x: str(cfg.IMAGES_DIR / x))

    # check non-existence file
    missing = [p for p in df["image_path"] if not Path(p).exists()]
    if missing:
        print(f"[WARNING] {len(missing)} image(s) not found. First 3:")
        for p in missing[:3]:
            print(f"{p}")
    else:
        print(f"[DATA] All {len(df)} image paths verified OK.")

    # remove Unnamed column
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    return df

In [4]:
def df_to_monai_list(df: pd.DataFrame) -> List[Dict]:
    return [
        {"image": row["image_path"], "label": int(row["label"])}
        for _, row in df.iterrows()
    ]

# Train test split

In [5]:
def patient_stratified_split(
    df: pd.DataFrame,
    val_size: float = 0.15,
    test_size: float = 0.15,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pat = (
        df.groupby(cfg.PATIENT_COL)["label"]
        .agg(lambda x: int(x.mean() >= 0.5))
        .reset_index()
    )

    p_tr, p_tmp, _, y_tmp = train_test_split(
        pat[cfg.PATIENT_COL].values,
        pat["label"].values,
        test_size=val_size + test_size,
        stratify=pat["label"].values,
        random_state=cfg.SEED,
    )
    p_val, p_test = train_test_split(
        p_tmp, test_size=0.5, stratify=y_tmp, random_state=cfg.SEED
    )

    tr = df[df[cfg.PATIENT_COL].isin(p_tr)].reset_index(drop=True)
    vl = df[df[cfg.PATIENT_COL].isin(p_val)].reset_index(drop=True)
    te = df[df[cfg.PATIENT_COL].isin(p_test)].reset_index(drop=True)

    for name, split in [("Train", tr), ("Val  ", vl), ("Test ", te)]:
        print(
            f"[SPLIT] {name}: {len(split):4d} images | "
            f"{split[cfg.PATIENT_COL].nunique():3d} patients | "
            f"GON+: {split['label'].sum():3d} | "
            f"GON-: {(split['label'] == 0).sum():3d}"
        )
    return tr, vl, te

# MONAI - Other Transformation

In [6]:
class FovMaskd(MapTransform):
    """
    Prevent model to focus on region outside fov by setting the pixel value of the region to 0.

    How it detects the FOV:
      1. Convert to uint8 grayscale
      2. Threshold at a low value to find any non-black pixel
      3. Find the largest contour (the FOV circle)
      4. Draw a filled circle mask from that contour's bounding geometry
      5. Apply the mask — pixels outside = 0.0
    """

    def __init__(self, keys: KeysCollection, threshold: int = 15):
        super().__init__(keys)
        self.threshold = threshold

    def __call__(self, data: dict) -> dict:
        d = dict(data)
        for key in self.keys:
            img = d[key]  # (C, H, W) float32

            # foce numpy datatype
            if hasattr(img, "numpy"):
                arr = img.numpy()
            else:
                arr = np.array(img)

            C, H, W = arr.shape

            # Find FOV contour
            gray = (arr.mean(axis=0) * 255).astype(np.uint8)  # (H, W)
            _, binary = cv2.threshold(gray, self.threshold, 255, cv2.THRESH_BINARY)
            binary = cv2.morphologyEx(
                binary,
                cv2.MORPH_CLOSE,
                kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)),
            )
            contours, _ = cv2.findContours(
                binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )

            # defining mask
            mask = np.zeros((H, W), dtype=np.uint8)
            if contours:
                largest = max(contours, key=cv2.contourArea)

                # fit a circle to the contour for a clean circular mask
                (cx, cy), radius = cv2.minEnclosingCircle(largest)
                cv2.circle(
                    mask,
                    center=(int(cx), int(cy)),
                    radius=int(radius),
                    color=255,
                    thickness=-1,
                )
            else:
                mask[:] = 255

            # apply mask
            mask_float = (mask / 255.0).astype(np.float32)  # (H, W), values 0 or 1
            arr = arr * mask_float[np.newaxis, :, :]  # broadcast over C

            # preserve original type
            if hasattr(img, "numpy"):
                d[key] = torch.from_numpy(arr)
            else:
                d[key] = arr

        return d

In [7]:
class CLAHEd(MapTransform):
    def __init__(
        self,
        keys: KeysCollection,
        clip_limit: float = 2.0,
        tile_grid_size: tuple = (8, 8),
    ):
        super().__init__(keys)
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, data: dict) -> dict:
        self.clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size
        )
        d = dict(data)
        for key in self.keys:
            img = d[key]

            if hasattr(img, "numpy"):
                arr = img.numpy()
                is_tensor = True
            else:
                arr = np.array(img)
                is_tensor = False

            # CLAHE operates on uint8
            arr_uint8 = (arr * 255).astype(np.uint8)  # (C, H, W)
            result = np.zeros_like(arr_uint8)

            for c in range(arr_uint8.shape[0]):
                result[c] = self.clahe.apply(arr_uint8[c])

            arr = result.astype(np.float32) / 255.0

            if is_tensor:
                d[key] = torch.from_numpy(arr)
            else:
                d[key] = arr

        return d


# MONAI Transform pipelines

In [8]:
def get_transforms(mode: str) -> Compose:
    transform_head = [
        LoadImaged(keys=["image"], image_only=True),
        EnsureChannelFirstd(keys=["image"]),
        ScaleIntensityd(keys=["image"], minv=0.0, maxv=1.0),
        Resized(
            keys=["image"], spatial_size=(cfg.IMG_SIZE, cfg.IMG_SIZE), mode="bicubic"
        ),
        FovMaskd(keys=["image"], threshold=10),  # remove black border signal
        CLAHEd(keys=["image"], clip_limit=2.0),  # boost disc interior contrast
    ]

    augment = [
        RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image"], prob=0.5, spatial_axis=1),
        RandRotate90d(keys=["image"], prob=0.5, max_k=3),
        RandRotated(
            keys=["image"],
            prob=0.4,
            range_x=0.52,
            range_y=0.0,
            range_z=0.0,
            mode="bilinear",
            padding_mode="reflection",
            keep_size=True,
        ),
        RandZoomd(
            keys=["image"], prob=0.3, min_zoom=0.85, max_zoom=1.15, mode="bilinear"
        ),
        RandAffined(
            keys=["image"],
            prob=0.3,
            translate_range=(0.05, 0.05),
            mode="bilinear",
            padding_mode="reflection",
        ),
        RandAdjustContrastd(keys=["image"], prob=0.4, gamma=(0.7, 1.5)),
        RandGaussianNoised(keys=["image"], prob=0.3, std=0.02),
        RandGaussianSmoothd(
            keys=["image"], prob=0.2, sigma_x=(0.5, 1.5), sigma_y=(0.5, 1.5)
        ),
    ]

    transform_tail = [
        NormalizeIntensityd(
            keys=["image"],
            subtrahend=[0.485, 0.456, 0.406],
            divisor=[0.229, 0.224, 0.225],
            channel_wise=True,
        ),
        ToTensord(keys=["image", "label"]),
    ]

    if mode == "train":
        return Compose(transform_head + augment + transform_tail)
    else:
        return Compose(transform_head + transform_tail)

In [9]:
def get_tta_transforms() -> List[Compose]:
    transform_head = [
        LoadImaged(keys=["image"], image_only=True, reader="pilreader"),
        EnsureChannelFirstd(keys=["image"]),
        ScaleIntensityd(keys=["image"], minv=0.0, maxv=1.0),
        Resized(
            keys=["image"], spatial_size=(cfg.IMG_SIZE, cfg.IMG_SIZE), mode="bicubic"
        ),
        FovMaskd(keys=["image"], threshold=10),  # remove black border signal
        CLAHEd(keys=["image"], clip_limit=2.0),  # boost disc interior contrast
    ]
    transform_tail = [
        NormalizeIntensityd(
            keys=["image"],
            subtrahend=[0.485, 0.456, 0.406],
            divisor=[0.229, 0.224, 0.225],
            channel_wise=True,
        ),
        ToTensord(keys=["image", "label"]),
    ]

    augment_ls = [
        [],
        [Flipd(keys=["image"], spatial_axis=0)],
        [Flipd(keys=["image"], spatial_axis=1)],
        [Rotate90d(keys=["image"], k=1)],
        [Rotate90d(keys=["image"], k=3)],
    ]
    return [Compose(transform_head + aug + transform_tail) for aug in augment_ls]

# Data loader


In [10]:
def make_loader(
    df: pd.DataFrame, transform: Compose, mode: str, sampler=None
) -> DataLoader:
    data_list = df_to_monai_list(df)
    cache = cfg.CACHE_RATE if mode != "train" else 0.0

    dataset = CacheDataset(
        data=data_list,
        transform=transform,
        cache_rate=cache,
        num_workers=cfg.NUM_WORKERS,
        copy_cache=False,
    )
    return DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=(mode == "train" and sampler is None),
        sampler=sampler,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=(mode == "train"),
    )

In [11]:
def make_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    counts = df["label"].value_counts().to_dict()
    total = len(df)
    weights = df["label"].map(lambda x: total / counts[x]).values
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=total,
        replacement=True,
    )

# Mix up

In [12]:
def mixup_data(x, y, alpha=cfg.MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

In [13]:
def mixup_loss(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya.float()) + (1 - lam) * criterion(pred, yb.float())

# MODEL


In [14]:
class GlaucomaModel(nn.Module):
    def __init__(
        self,
        backbone: str = cfg.BACKBONE,
        pretrained: bool = cfg.PRETRAINED,
        dropout: float = cfg.DROPOUT,
    ):
        super().__init__()
        self.backbone = timm.create_model(
            backbone,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )
        in_features = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x)).squeeze(1)

# Training Components

In [15]:
class EarlyStopping:
    def __init__(self, patience=cfg.EARLY_STOP, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best = None
        self.stop = False

    def __call__(self, score: float) -> bool:
        if self.best is None or score > self.best + self.min_delta:
            self.best = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

In [16]:
def train_one_epoch(
    model: GlaucomaModel, loader: DataLoader, optimizer, criterion
) -> Dict:
    model.train()
    total_loss, all_probs, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc="Train", leave=False):
        imgs = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        if cfg.USE_MIXUP:
            imgs, ya, yb, lam = mixup_data(imgs, labels)

        logits = model(imgs)
        loss = (
            mixup_loss(criterion, logits, ya, yb, lam)
            if cfg.USE_MIXUP
            else criterion(logits, labels.float())
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    preds_bin = (np.array(all_probs) >= 0.5).astype(int)
    return {
        "loss": total_loss / len(loader.dataset),
        "auc": roc_auc_score(all_labels, all_probs),
        "acc": accuracy_score(all_labels, preds_bin),
    }

In [17]:
@torch.no_grad()
def evaluate(model: GlaucomaModel, loader: DataLoader, criterion) -> Dict:
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []

    for batch in tqdm(loader, desc="Eval", leave=False):
        imgs = batch["image"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        logits = model(imgs)
        loss = criterion(logits, labels.float())

        total_loss += loss.item() * imgs.size(0)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    preds_bin = (np.array(all_probs) >= 0.5).astype(int)
    return {
        "loss": total_loss / len(loader.dataset),
        "auc": roc_auc_score(all_labels, all_probs),
        "acc": accuracy_score(all_labels, preds_bin),
        "f1": f1_score(all_labels, preds_bin),
        "preds": np.array(all_probs),
        "labels": np.array(all_labels),
    }

In [18]:
def train(train_df, val_df, fold=None) -> GlaucomaModel:
    tag = f"fold{fold}" if fold is not None else "model"
    ckpt_path = cfg.OUTPUT_DIR / f"best_{tag}.pt"

    sampler = make_weighted_sampler(train_df)
    train_loader = make_loader(
        train_df, get_transforms("train"), mode="train", sampler=sampler
    )
    val_loader = make_loader(val_df, get_transforms("val"), mode="val")

    model = GlaucomaModel().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [(train_df["label"] == 0).sum() / max((train_df["label"] == 1).sum(), 1)],
            device=DEVICE,
        )
    )
    optimizer = optim.AdamW(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.EPOCHS, eta_min=1e-6
    )
    stopper = EarlyStopping()
    best_auc = 0.0
    history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": []}

    print(f"\n{'=' * 60}\n  {tag.upper()}  —  {cfg.BACKBONE}\n{'=' * 60}")

    for epoch in range(1, cfg.EPOCHS + 1):
        tr = train_one_epoch(model, train_loader, optimizer, criterion)
        vl = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tr["loss"])
        history["val_loss"].append(vl["loss"])
        history["train_auc"].append(tr["auc"])
        history["val_auc"].append(vl["auc"])

        flag = ""
        if vl["auc"] > best_auc:
            best_auc = vl["auc"]
            torch.save(model.state_dict(), ckpt_path)
            flag = "  ← best"

        print(
            f"  Ep {epoch:03d}/{cfg.EPOCHS} "
            f"| Loss {tr['loss']:.4f}/{vl['loss']:.4f} "
            f"| AUC {tr['auc']:.4f}/{vl['auc']:.4f} "
            f"| Acc {tr['acc']:.4f}/{vl['acc']:.4f}{flag}"
        )

        if stopper(vl["auc"]):
            print(f"  [EARLY STOP] No gain for {cfg.EARLY_STOP} epochs.")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return model, history

# TTA Prediction

In [19]:
@torch.no_grad()
def tta_predict(model: GlaucomaModel, df: pd.DataFrame) -> np.ndarray:
    model.eval()
    tta_tfms = get_tta_transforms()[: cfg.TTA_STEPS]
    all_probs = []

    for tfm in tqdm(tta_tfms, desc="TTA"):
        loader = make_loader(df, tfm, mode="test")
        probs = []
        for batch in loader:
            logits = model(batch["image"].to(DEVICE))
            probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_probs.append(np.array(probs))

    return np.mean(all_probs, axis=0)

In [20]:
def find_optimal_threshold(labels: np.ndarray, probs: np.ndarray) -> float:
    fpr, tpr, thresholds = roc_curve(labels, probs)
    return float(thresholds[np.argmax(tpr - fpr)])

In [21]:
def full_evaluation(labels, probs, split="Test") -> Tuple[Dict, np.ndarray, np.ndarray]:
    threshold = find_optimal_threshold(labels, probs)
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "AUC-ROC": roc_auc_score(labels, probs),
        "Accuracy": accuracy_score(labels, preds),
        "F1 (macro)": f1_score(labels, preds, average="macro"),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "PPV": tp / (tp + fp),
        "NPV": tn / (tn + fn),
        "Threshold": threshold,
    }

    print(f"\n{'=' * 50}\n  {split} Results(threshold = {threshold:.3f})\n{'=' * 52}")
    for k, v in metrics.items():
        print(f"  {k:<14}: {v:.4f}")
    print(f"\n  Confusion Matrix:\n{cm}")
    print(f"\n{classification_report(labels, preds, target_names=cfg.CLASS_NAMES)}")
    return metrics, preds, cm

# Main

In [22]:
# load dataset
df = load_and_clean_labels(cfg.LABELS_CSV)
train_df, val_df, test_df = patient_stratified_split(df)

[DATA] Raw: 747 images, 288 patients
Classes:
Label
GON+    548
GON-    199
Name: count, dtype: int64

[DATA] All 747 image paths verified OK.
[SPLIT] Train:  502 images | 201 patients | GON+: 363 | GON-: 139
[SPLIT] Val  :  124 images |  43 patients | GON+:  93 | GON-:  31
[SPLIT] Test :  121 images |  44 patients | GON+:  92 | GON-:  29


In [ ]:
# Train model
model, history = train(train_df, val_df)
torch.save(model.state_dict(), cfg.FINAL_OUTPUT_DIR / "final_model.pt")
json.dump(history, open(cfg.FINAL_OUTPUT_DIR / "training_history.json", "w"))

Loading dataset: 100%|██████████| 124/124 [00:08<00:00, 15.02it/s]



  MODEL  —  efficientnet_b3


Train:   0%|          | 0/31 [00:00<?, ?it/s]

In [ ]:
# DISABLE AVOVE CELL AND RUN THIS CELL ONLY TO SKIP TRAINING STEP
# DONT RUN THIS CELL IF YOU NEED TRAINING

# save results - model
# model = GlaucomaModel()
# model.load_state_dict(torch.load(cfg.OUTPUT_DIR / "best_model.pt", map_location=DEVICE))
# torch.save(model.state_dict(), cfg.FINAL_OUTPUT_DIR / "final_model.pt")

# save result
# history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": []}
# json.dump(history, open(cfg.FINAL_OUTPUT_DIR / "training_history.json", "w"))